# Programatically run an LLM for data analysis
## Get LLM Started


In [ ]:
#### Create copy to use later:
# Click File > Save a Copy in Drive
# Close this notebook, work from "Copy of LLM..."
# In "Copy of LLM...":
# Click Connect (top-right) > click "RunTime" OR "change runtime type" > select "T4"
# LLM is running when you see tick & "RAM / Disk" icons

In [ ]:
#### Load AI libraries:
# Load libraries supporting AI analysis
!pip install PyMuPDF --quiet
!pip install tools --quiet
!pip install accelerate --quiet
!pip install -U bitsandbytes --quiet
!pip install transformers --quiet

import json
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
import warnings
import requests
import fitz  # PyMuPDF
import pandas as pd
from io import BytesIO
import urllib.parse

In [ ]:
#### Load AI model
# Load Large Language Model (LLM)
model_id = "Qwen/Qwen2.5-3B"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16
)

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto"
)

tokenizer = AutoTokenizer.from_pretrained(model_id)
print("--- Model Loaded Successfully ---")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

The fast path is not available because one of the required library is not installed. Falling back to torch implementation. To install follow https://github.com/fla-org/flash-linear-attention#installation and https://github.com/Dao-AILab/causal-conv1d


Loading weights:   0%|          | 0/426 [00:00<?, ?it/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/12.8M [00:00<?, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

--- Model Loaded Successfully ---


## Data

In [ ]:
#### Load Parliamentary data
quotes = pd.read_csv("https://raw.githubusercontent.com/drhemming/LLM-1h-workshop-repository/main/Parliamentary_data/hansard_quotes_summer_llm_workshop_march.csv", encoding='latin1')

# Show column names of data
print("Column names:")
print(quotes.columns)

Column names:
Index(['Quote', 'Speaker'], dtype='object')


In [ ]:
# Display one quote of the data set
print(quotes.at[0, "Quote"])

The context of this bill is that this bill seeks to disestablish the Ministry for the Environment. It folds, and it folds into a mega-ministry alongside Transport, Housing, Urban Development, and parts of Internal Affairs. I think itÕs bizarre that now more than ever, we need to prioritise the environment first and foremost. We cannot deny climate change, and when we look at the bigger picture and focus on climate change and the environment, we cannot deny the weather events that weÕve seen across the country in the past month. In just the first weeks of 2026, there has been eight states of emergency that have been declared, matching the total for all of 2025. Recent storms across to Te Ika-a-M_uiÑthe North Island have brought record rainfall, flooding, evacuations, power outages, and loss of life. January storms alone caused multiple deaths from landslides and flooding, with record daily rainfall and Tauranga and Whitianga.


In [ ]:
# Display the first rows of data
print("First 5 quotes:")
print(quotes.head())

First 5 quotes:
                                               Quote  \
0  The context of this bill is that this bill see...   
1  Equally, in terms of that scrutiny function, i...   
2  I remember being part of the Environment Commi...   
3  Point of order. Madam Speaker, this latest dia...   
4  Thank you, Madam Speaker. I rise to take a cal...   

                     Speaker  
0  HANA-RAWHITI MAIPI-CLARKE  
1         Hon Dr DUNCAN WEBB  
2               GLEN BENNETT  
3      Rt Hon Winston Peters  
4             GRANT McCALLUM  


In [ ]:
# Test run LLM on first quote (row 1 of data)


#### Test run LLM on first quote (row 1 of data)

## Prompts

In [ ]:
# prompts
quotes['main_theme'] = ''
quotes['political_spectrum'] = ''
quotes['theme_justification']=''
quotes['spectrum_justification']=''

# Analyse data and stick analysis into each new column cell
for row in quotes.itertuples(index = True):
    quote = row.Quote
    speaker = row.Speaker

    # Construct the LLM prompt for row-by-row analysis
    prompt_row_by_row = [
        {
            "role": "system",
            "content": "You are an expert qualitative research assistant specializing in New Zealand parliamentary debates (Hansard data). Your task is to analyze a single quote and speaker and extract specific information in a strict JSON format."
        },
        {
            "role": "user",
            "content": f"""
                       The below message is text extracted from a new zealand parliament debate. More specifically, it is from one question within the debate.
                       Each line contains a quote from a speaker.
                       The JSON should contain the following fields:
                       main_theme: provide the main theme of the quote presented as a short (one to three word) phrase.
                       theme_justification: provide an alternative theme to the main theme identified for each quote, one that a researcher with a different political bias might find.
                       political_spectrum: categorize the political orientation of the quote using the New Zealand political spectrum: far-left, left-wing, center-left, centrist, center-right, right-wing or far-right.
                       spectrum_justification: for the political sprectrum identified, state which specific New Zealand policy platform (e.g., 'The Clean Car Discount', 'Three Waters', or 'Tax Bracket Indexation') this rhetoric aligns with. If it doesn't align with a specific policy, explain why you chose that spectrum position.

                        --- Speaker ---

                       {row.Speaker}

                        --- QUOTE ---

                        {row.Quote}

                        --- END QUOTE ---

                        Do not include comments in your JSON response. Only respond with the JSON object. Make sure the JSON is valid

                        """,
        }
    ]

    # Tokenize and Run Inference for the current row
    chat_input_row = tokenizer.apply_chat_template(prompt_row_by_row, tokenize=False, add_generation_prompt=True)

    tokenized_output_row = tokenizer(
        chat_input_row,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=tokenizer.model_max_length # Ensure truncation for long inputs
    )

    input_ids_row = tokenized_output_row['input_ids'].to(model.device)
    attention_mask_row = tokenized_output_row['attention_mask'].to(model.device)

    generated_ids_row = model.generate(
        input_ids=input_ids_row,
        attention_mask=attention_mask_row,
        max_new_tokens=5000,
        pad_token_id=tokenizer.eos_token_id
    )

    decoded_generated_text_row = tokenizer.batch_decode(
        generated_ids_row[:, input_ids_row.shape[-1]:],
        skip_special_tokens=True,
        clean_up_tokenization_spaces=False
    )[0]

    raw_output_string_row = decoded_generated_text_row.strip()

    # Attempt to parse JSON
    decoder = json.JSONDecoder()
    result_json, end_index = decoder.raw_decode(raw_output_string_row)

    # Show row-by-row outputs
    print("Main Theme:", result_json['main_theme'], "| Theme Justification:", result_json['theme_justification'], "| Political Spectrum:", result_json['political_spectrum'], "| Spectrum Justification:", result_json['spectrum_justification'])

    # Display data
    quotes.loc[row.Index, 'main_theme'] = result_json['main_theme']
    quotes.loc[row.Index, 'theme_justification'] = result_json['theme_justification']
    quotes.loc[row.Index, 'political_spectrum'] = result_json['political_spectrum']
    quotes.loc[row.Index, 'spectrum_justification'] = result_json['spectrum_justification']

KeyboardInterrupt: 

# Outputs

In [ ]:
# Updated quotes data frame
quotes

,Quote,Speaker,main_theme,political_spectrum,theme_justification,spectrum_justification
0,The context of this bill is that this bill see...,HANA-RAWHITI MAIPI-CLARKE,environmental protection,center-left,climate action,This rhetoric aligns with the center-left poli...
1,"Equally, in terms of that scrutiny function, i...",Hon Dr DUNCAN WEBB,scrutiny,center-right,regulation quality,The speech reflects a common stance among cent...
2,I remember being part of the Environment Commi...,GLEN BENNETT,environment protection,center-right,public opposition to environmental legislation,This quote aligns with the center-right politi...
3,"Point of order. Madam Speaker, this latest dia...",Rt Hon Winston Peters,topic diversion,center-right,side-taking,This speech does not align with a specific pol...
4,"Thank you, Madam Speaker. I rise to take a cal...",GRANT McCALLUM,environmental integration,center-right,environmental oversight consolidation,This theme aligns with center-right policies t...
5,"Point of order. Thank you, Madam Speaker. Look...",Simon Court,duplicitous accusations,center-right,misrepresenting bills,This theme aligns with the center-right politi...
6,"That is, as far as IÕm aware, a sitting week. ...",RICARDO MENNDEZ MARCH,scrutiny,center-right,legislative oversight,This speech reflects a balanced view of legisl...
7,This will support faster decisions and better ...,Hon TAMA POTAKA,policy coordination,center-right,policy fragmentation,This speech does not align with any specific N...
8,"Now, I want to acknowledge, as well, those lea...",Lan Pham,environmental protest,far-left,environmental critique,The speaker criticizes the National Party's po...


## Prompts, Outputs - more advanced

In [ ]:
# new columns:
quotes['main_theme'] = ''
quotes['political_spectrum'] = ''

quotes['prompt_5'] = ''
quotes['prompt_6'] = ''

# Analyse data and stick analysis into each new column cell
for row in quotes.itertuples(index = True):
    quote = row.Quote
    speaker = row.Speaker

    # Construct the LLM prompt for row-by-row analysis
    prompt_row_by_row = [
        {
            "role": "system",
            "content": "You are an expert qualitative research assistant specializing in New Zealand parliamentary debates (Hansard data). Your task is to analyze a single quote and speaker and extract specific information in a strict JSON format."
        },
        {
            "role": "user",
            "content": f"""
                       The below message is text extracted from a new zealand parliament debate. More specifically, it is from one question within the debate.
                       Each line contains a quote from a speaker.
                       The JSON should contain the following fields:
                       main_theme: provide the main theme of the quote presented as a short (one to three word) phrase.
                       political_spectrum: categorize the political orientation of the quote using the New Zealand political spectrum: far-left, teft-wing, center-left, centrist, center-right, right-wing or far-right.
                       prompt_5: infer how many hotdogs the speaker could consume in one sitting.
                       prompt_6: justfiy your reason why the speaker could only consume that many hot dogs.
                        --- Speaker ---

                       {row.Speaker}

                        --- QUOTE ---

                        {row.Quote}

                        --- END QUOTE ---

                        Do not include comments in your JSON response. Only respond with the JSON object. Make sure the JSON is valid

                        """,
        }
    ]

    # Tokenize and Run Inference for the current row
    chat_input_row = tokenizer.apply_chat_template(prompt_row_by_row, tokenize=False, add_generation_prompt=True)

    tokenized_output_row = tokenizer(
        chat_input_row,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=tokenizer.model_max_length # Ensure truncation for long inputs
    )

    input_ids_row = tokenized_output_row['input_ids'].to(model.device)
    attention_mask_row = tokenized_output_row['attention_mask'].to(model.device)

    generated_ids_row = model.generate(
        input_ids=input_ids_row,
        attention_mask=attention_mask_row,
        max_new_tokens=5000,
        pad_token_id=tokenizer.eos_token_id
    )

    decoded_generated_text_row = tokenizer.batch_decode(
        generated_ids_row[:, input_ids_row.shape[-1]:],
        skip_special_tokens=True,
        clean_up_tokenization_spaces=False
    )[0]

    raw_output_string_row = decoded_generated_text_row.strip()

    # Attempt to parse JSON
    decoder = json.JSONDecoder()
    result_json, end_index = decoder.raw_decode(raw_output_string_row)

    # Show row-by-row outputs
    print("Main Theme:", result_json['main_theme'], "| Political Spectrum:", result_json['political_spectrum'], "| Prompt 5:", result_json['prompt_5'], "| Prompt 6:", result_json['prompt_6'])

    # Display data
    quotes.loc[row.Index, 'main_theme'] = result_json['main_theme']
    quotes.loc[row.Index, 'political_spectrum'] = result_json['political_spectrum']
    quotes.loc[row.Index, 'prompt_5'] = result_json['prompt_5']
    quotes.loc[row.Index, 'prompt_6'] = result_json['prompt_6']

Main Theme: environmental priorities | Political Spectrum: center-left | Prompt 5: 1000 | Prompt 6: The speaker could consume 1000 hot dogs in one sitting because they emphasize the importance of prioritizing environmental issues, which suggests a strong stomach capacity.
Main Theme: scrutiny | Political Spectrum: center-right | Prompt 5: 30 | Prompt 6: The speaker cannot provide a definitive number as they do not have actual information about the Minister's consumption habits. However, they use this hypothetical example to emphasize the lack of thorough scrutiny given the government's purported commitment to regulatory quality.
Main Theme: environment protection | Political Spectrum: center-right | Prompt 5: 30000 | Prompt 6: The speaker consumed a large quantity of hot dogs because they were passionate about the environmental cause and the scale of public support demonstrated by the 30,000 people marching on Queen Street.
Main Theme: side debate | Political Spectrum: center-right | P

In [ ]:
# new columns:
quotes['prompt_1'] = ''
quotes['prompt_2'] = ''
quotes['prompt_3'] = ''
quotes['prompt_4'] = ''

# Analyse data and stick analysis into each new column cell
for row in quotes.itertuples(index = True):
    quote = row.Quote
    speaker = row.Speaker

    # Construct the LLM prompt for row-by-row analysis
    prompt_row_by_row = [
        {
            "role": "system",
            "content": "You are an expert qualitative research assistant specializing in New Zealand parliamentary debates (Hansard data). Your task is to analyze a single quote and speaker and extract specific information in a strict JSON format."
        },
        {
            "role": "user",
            "content": f"""
                       The below message is text extracted from a new zealand parliament debate. More specifically, it is from one question within the debate.
                       Each line contains a quote from a speaker.
                       The JSON should contain the following fields:
                       prompt_1: How well would this person survive on Mars? From 1 to 100.
                       prompt_2: How many balloons would it take to carry the speaker away to the moon?
                       prompt_3: Why did the speaker cross the road?
                       prompt_4: What would their cat name be?

                        --- Speaker ---

                       {row.Speaker}

                        --- QUOTE ---

                        {row.Quote}

                        --- END QUOTE ---

                        Do not include comments in your JSON response. Only respond with the JSON object. Make sure the JSON is valid

                        """,
        }
    ]

    # Tokenize and Run Inference for the current row
    chat_input_row = tokenizer.apply_chat_template(prompt_row_by_row, tokenize=False, add_generation_prompt=True)

    tokenized_output_row = tokenizer(
        chat_input_row,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=tokenizer.model_max_length # Ensure truncation for long inputs
    )

    input_ids_row = tokenized_output_row['input_ids'].to(model.device)
    attention_mask_row = tokenized_output_row['attention_mask'].to(model.device)

    generated_ids_row = model.generate(
        input_ids=input_ids_row,
        attention_mask=attention_mask_row,
        max_new_tokens=5000,
        pad_token_id=tokenizer.eos_token_id
    )

    decoded_generated_text_row = tokenizer.batch_decode(
        generated_ids_row[:, input_ids_row.shape[-1]:],
        skip_special_tokens=True,
        clean_up_tokenization_spaces=False
    )[0]

    raw_output_string_row = decoded_generated_text_row.strip()

    # Attempt to parse JSON
    decoder = json.JSONDecoder()
    result_json, end_index = decoder.raw_decode(raw_output_string_row)

    # Show row-by-row outputs
    print("Prompt 1:", result_json[speaker]['prompt_1'], "Prompt 2:", result_json[speaker]['prompt_2'], "Prompt 3:", result_json[speaker]['prompt_3'], "Prompt 4:", result_json[speaker]['prompt_4'],)

    # Display data
    quotes.loc[row.Index, 'prompt_1'] = result_json[speaker]['prompt_1']
    quotes.loc[row.Index, 'prompt_2'] = result_json[speaker]['prompt_2']
    quotes.loc[row.Index, 'prompt_3'] = result_json[speaker]['prompt_3']
    quotes.loc[row.Index, 'prompt_4'] = result_json[speaker]['prompt_4']

Prompt 1: How well would this person survive on Mars? From 1 to 100. Prompt 2: How many balloons would it take to carry the speaker away to the moon? Prompt 3: Why did the speaker cross the road? Prompt 4: What would their cat name be?
Prompt 1: None Prompt 2: None Prompt 3: None Prompt 4: None


KeyError: 'prompt_1'

# Prompting "Validated" Outputs

In [ ]:
# new columns:
quotes['main_theme'] = ''
quotes['theme_justification'] = ''

quotes['political_spectrum'] = ''
quotes['spectrum_justification'] = ''

# Analyse data and stick analysis into each new column cell
for row in quotes.itertuples(index = True):
    quote = row.Quote
    speaker = row.Speaker

    # Construct the LLM prompt for row-by-row analysis
    prompt_row_by_row = [
        {
            "role": "system",
            "content": "You are an expert qualitative research assistant specializing in New Zealand parliamentary debates (Hansard data). Your task is to analyze a single quote and speaker and extract specific information in a strict JSON format."
        },
        {
            "role": "user",
            "content": f"""
                       The below message is text extracted from a new zealand parliament debate. More specifically, it is from one question within the debate.
                       Each line contains a quote from a speaker.
                       The JSON should contain the following fields:
                       main_theme: provide the main theme of the quote presented as a short (one to three word) phrase.
                       theme_justification: provide an alternative theme to the main theme identified for each quote, one that a researcher with a different political bias might find.
                       political_spectrum: categorize the political orientation of the quote using the New Zealand political spectrum: far-left, teft-wing, center-left, centrist, center-right, right-wing or far-right.
                       spectrum_justification: for the political sprectrum identified, state which specific New Zealand policy platform (e.g., 'The Clean Car Discount', 'Three Waters', or 'Tax Bracket Indexation') this rhetoric aligns with. If it doesn't align with a specific policy, explain why you chose that spectrum position.

                        --- Speaker ---

                       {row.Speaker}

                        --- QUOTE ---

                        {row.Quote}

                        --- END QUOTE ---

                        Do not include comments in your JSON response. Only respond with the JSON object. Make sure the JSON is valid

                        """,
        }
    ]

    # Tokenize and Run Inference for the current row
    chat_input_row = tokenizer.apply_chat_template(prompt_row_by_row, tokenize=False, add_generation_prompt=True)

    tokenized_output_row = tokenizer(
        chat_input_row,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=tokenizer.model_max_length # Ensure truncation for long inputs
    )

    input_ids_row = tokenized_output_row['input_ids'].to(model.device)
    attention_mask_row = tokenized_output_row['attention_mask'].to(model.device)

    generated_ids_row = model.generate(
        input_ids=input_ids_row,
        attention_mask=attention_mask_row,
        max_new_tokens=5000,
        pad_token_id=tokenizer.eos_token_id
    )

    decoded_generated_text_row = tokenizer.batch_decode(
        generated_ids_row[:, input_ids_row.shape[-1]:],
        skip_special_tokens=True,
        clean_up_tokenization_spaces=False
    )[0]

    raw_output_string_row = decoded_generated_text_row.strip()

    # Attempt to parse JSON
    decoder = json.JSONDecoder()
    result_json, end_index = decoder.raw_decode(raw_output_string_row)

    # Show row-by-row outputs
    print("Main Theme:", result_json['main_theme'], "| Theme Justification:", result_json['theme_justification'], "| Political Spectrum:", result_json['political_spectrum'], "| Spectrum Justification:", result_json['spectrum_justification'])

    # Display data
    quotes.loc[row.Index, 'main_theme'] = result_json['main_theme']
    quotes.loc[row.Index, 'theme_justification'] = result_json['theme_justification']
    quotes.loc[row.Index, 'political_spectrum'] = result_json['political_spectrum']
    quotes.loc[row.Index, 'spectrum_justification'] = result_json['spectrum_justification']

Main Theme: environmental prioritization | Theme Justification: environmental deregulation | Political Spectrum: center-left | Spectrum Justification: This quote aligns with center-left policies as it emphasizes environmental protection but does not directly support specific policy platforms like The Clean Car Discount or Three Waters.
Main Theme: scrutiny | Theme Justification: regulatory transparency | Political Spectrum: center-right | Spectrum Justification: This quote aligns with center-right political values by emphasizing the importance of regulatory quality and thorough scrutiny, which is often seen as a conservative stance.
Main Theme: environmental protection | Theme Justification: public opposition to environmental legislation | Political Spectrum: center-left | Spectrum Justification: This quote aligns with the center-left spectrum as it reflects a common public sentiment against environmentally harmful policies, which is often a concern for left-leaning politicians.
Main T

# What on earth is going on in this script?

Below is a annotated copy of two important of the important cells that have been introduced in this workshop:
1) Model Loading
2) Model running

There is quite a bit going on in this code, so the aim here is to help you understand more about what each line of the code is doing.

For our LLM, we have used the HuggingFace library. This is a high-level API built on top of PyTorch, a fundamental deep learning library developed by Meta that allows you to build all kinds of models for computer vision, natural language processing, reinforcement learning, etc.

Hugginface is really great and has loads of models to chose from. Theres also courses to dive deeper into a specific aspect of deep learning.

Everything that is discussed below can be found on Huggingface. The model we used - Qwen2.5-3B-Instruct - has good documentation that provided the backbone for this workshop. You can take a look [here](https://huggingface.co/Qwen/Qwen2.5-3B-Instruct). Similarly, if you want to use a different model, you can look at the relevant docs for that model on huggingface for some handy info on how to instantiate it.

We will start with loading the model.

In [ ]:
# Loading the model

# First we specify what model we want. The list of models can be found here (https://huggingface.co/models).
model_id = "Qwen/Qwen2.5-3B-Instruct"

# This step is 'optional' for some models, but necessary for others. It is called 'Quantization'.
# Quantization reduces the memory and computational cost of a model by representing its weights with lower precision. You can think of this essentially as reducing the number of decimal places.
# The Qwen model we are using here uses 'BF16' (16 Bit floating point)by default. This is precise, but would eat up all of our RAM. We can either use a smaller model, or quantize the current.
# We first start by defininig our quantization module that we will apply when we load in our model. We will use the BitsAndBytesConfig module for this.

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True, # Specify that we want to load the model in 4bit. Reduces memory usage by upto 4x.
    bnb_4bit_quant_type="nf4", # We specify the quantization type, always nf4.
    bnb_4bit_use_double_quant=True, # This saves even more memory.
    bnb_4bit_compute_dtype=torch.bfloat16 # Default is float32, setting to bfloat16 speeds up computation.
)

# Now we are actually loading in our model. We use the class 'AutoModel'. There are a number of different types of 'AutoModel' classes.
model = AutoModelForCausalLM.from_pretrained(  # We chose 'AutoModelForCausalLM'. You can find which one to use by looking at the relevant docs for your model on huggingface.co
    model_id, # We specify the name of the model we want.
    quantization_config=bnb_config, # We specify how we want to quantize the model using the object we've just created.
    device_map="auto" # We specify what 'device' (CPU or GPU) we want the model to be on. 'Auto' says, pick the fastest available device.
)

tokenizer = AutoTokenizer.from_pretrained(model_id) # We now load our tokenizer. This converts our prompts into numeric. Again, see the relevant docs for the model on huggingface.co.
print("--- Model Loaded Successfully ---")

And now we move onto writing our prompt and passing it through the model to generate an output.




In [ ]:
# We are adding new columns that we will add the results of prompt to

# Add some new columns based on what we want the prompt to generate.
quotes['speaker_party'] = ''
quotes['main_topic'] = ''
quotes['argumentative_rating'] = ''

# Now we define our main for loop. This will loop through every row in our dataframe, applying the prompt and generating output for each.
for row in df_year.itertuples(index = True):
    quote = row.Quote # MIGHT BE ABLE TO REMOVE THESE
    speaker = row.Speaker # MIGHT BE ABLE TO REMOVE THESE

    # Now we create our prompt. It contains a system prompt that gives the LLM some context as to how we want it to act, and a user prompt that is our question.
    prompt_row_by_row = [
        { # The system prompt. We are asking to LLM to act a certain way, or to adopt a persona that will inform how it responds.
            "role": "system",
            "content": "You are an expert qualitative research assistant specializing in New Zealand parliamentary debates (Hansard data). Your task is to analyze a single quote and speaker and extract specific information in a strict JSON format."
        },
        { # The user prompt. This is the question we are asking the LLM.
            "role": "user",
            "content": f"""
                       The below message is text extracted from a new zealand parliament debate. More specifically, it is from one question within the debate.
                       Each line contains a quote from a speaker.
                       The JSON should contain the following fields:
                       speaker_party: the political party of the speaker.
                       main_topic: summarise the topic of quote.
                       argumentative_rating: give the speaker a rating of how argumentative they are from 1 to 10.

                        --- Speaker ---

                       {row.Speaker}

                        --- QUOTE ---

                        {row.Quote}

                        --- END QUOTE ---

                        Do not include comments in your JSON response. Only respond with the JSON object. Make sure the JSON is valid

                        """,
        }
    ]


    # Apply chat template if you are working with multiple prompts to ensure the correct format. Make sure to not tokenize our input just yet.
    # Add generation prompt tells the model to answer after the prompt is finished.
    chat_input_row = tokenizer.apply_chat_template(prompt_row_by_row, tokenize=False, add_generation_prompt=True)


    # Now we tokenize our input, converting from string to numeric
    tokenized_output_row = tokenizer( # call our tokenizer class that we instantiated earlier
        chat_input_row,  # pass in our correctly formatted prompt
        return_tensors="pt", # Ensure we use the correct format for our tensors. 'pt' = pytorch. Huggingface also works with Jax ('jax'), tensorflow ('tf') and numpy ('np'), but we mostly use pytorch.
        padding=True, # Padding ensures all sequences are the same length.
        truncation=True, # Truncating does a similar thing, shortens long sequences so all are the same length.
        max_length=tokenizer.model_max_length # Ensure truncation for long inputs
    )

    # We now have to send a view things to our device (CPU or GPU)
    input_ids_row = tokenized_output_row['input_ids'].to(model.device) # Input ids refer to our 'tokens', AKA our words (or however the model decides to generate tokens).
    attention_mask_row = tokenized_output_row['attention_mask'].to(model.device) # attention mask gives each input id a number which denotes whether the model should attend to it or not. Because we have padded some sequences, we need this.


    # We now use our model to generate an output. However, this output will be a tokenized output (i.e., numerical), so its not really readable yet.
    generated_ids_row = model.generate(
        input_ids=input_ids_row, # Specify our tokens (i.e., our prompt)
        attention_mask=attention_mask_row, # Specify our attention mask (i.e., what tokens to attend to)
        max_new_tokens=256, # Adjust max_new_tokens for expected JSON output length - This adjusts how big the answers will be
        pad_token_id=tokenizer.eos_token_id # This is a special token that tells the model to stop generating output. Otherwise, it will keep on generating until it hits 'max_new_tokens'.
    )

    # Now we need to convert our generated output from tokens (numeric) back to text strings that we can read
    decoded_generated_text_row = tokenizer.batch_decode( # Batch decode used to decode back to strings. This includes both our original prompt and the generated output
        generated_ids_row[:, input_ids_row.shape[-1]:], # This ensures we only get the generated output.
        skip_special_tokens=True, # Removes markers (special tokens) such as <|endoftext|>
        clean_up_tokenization_spaces=False # Preferred setting, stops LLM trying to format text oddly
    )[0]

    raw_output_string_row = decoded_generated_text_row.strip() # Format our strings to remove any leading or trailing whitespaces

    # Our output is in json format, we need to now decode that
    decoder = json.JSONDecoder()
    result_json, end_index = decoder.raw_decode(raw_output_string_row) # We only really need the 'result_json' here. The 'end_index' is the position in 'raw_output_string_row' where decoding finished.

    # Show row-by-row outputs
    print("Party:", result_json['speaker_party'], "| Topic:", result_json['main_topic'], "| Argumentative rating:", result_json['argumentative_rating'])

    # Update the DataFrame with inferred values
    quotes.loc[row.Index, 'speaker_party'] = result_json['speaker_party']
    quotes.loc[row.Index, 'main_topic'] = result_json['main_topic']
    quotes.loc[row.Index, 'argumentative_rating'] = result_json['argumentative_rating']

